In [1]:
library(keras)
library(tensorflow)
library(tidyverse)
library(recipes)
set.seed(2) 

Warning message:
"le package 'keras' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tensorflow' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tidyverse' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'ggplot2' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tibble' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tidyr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'readr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'dplyr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'forcats' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'lubridate' a été compilé avec la version R 4.2.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.2     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.2     ✔ tibble    3.2.1
✔ lubridate 1.9.2    

In [2]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

In [3]:
data<-read.csv("train_values.csv",stringsAsFactors = T)
data_labels<-read.csv("train_labels.csv",stringsAsFactors = T)
datam<-merge(data,data_labels,by=c('building_id','building_id'))

id_variable <- match('building_id', colnames(datam))

In [43]:
head(datam)

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,⋯,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade,geo_level_1_mean_damage,geo_level_1_mean_damage.1,after
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<fct>,<fct>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<lgl>,<lgl>,<chr>
1,4,30,266,1224,1,25,5,2,t,r,⋯,0,0,0,0,0,0,2,NA,NA,geo_level_1_id
2,8,17,409,12182,2,0,13,7,t,r,⋯,0,0,0,0,0,0,3,NA,NA,geo_level_1_id
3,12,17,716,7056,2,5,12,6,o,r,⋯,0,0,0,0,0,0,3,NA,NA,geo_level_1_id
4,16,4,651,105,2,80,5,4,n,r,⋯,0,0,0,0,0,0,2,NA,NA,geo_level_1_id
5,17,3,1387,3909,5,40,5,10,t,r,⋯,0,0,0,0,0,0,2,NA,NA,geo_level_1_id
6,25,26,1132,6645,2,0,6,6,t,w,⋯,0,0,0,0,0,0,1,NA,NA,geo_level_1_id


In [51]:
target_variable<-match('damage_grade', colnames(datam))
id_loc_1 <- match('geo_level_1_id', colnames(datam))
id_loc_2 <- match('geo_level_2_id', colnames(datam))
id_loc_3 <- match('geo_level_3_id', colnames(datam))
id_loc_1_vals<-sort(unique(datam[,id_loc_1]))
id_loc_2_vals<-sort(unique(datam[,id_loc_2]))
id_loc_3_vals<-sort(unique(datam[,id_loc_3]))
id_loc_1_df <- data.frame(matrix(ncol = 2, nrow = length(id_loc_1_vals)))
colnames(id_loc_1_df)<-c('id','mean')
id_loc_2_df <- data.frame(matrix(ncol = 2, nrow = length(id_loc_2_vals)))
colnames(id_loc_2_df)<-c('id','mean')
id_loc_3_df <- data.frame(matrix(ncol = 2, nrow = length(id_loc_3_vals)))
colnames(id_loc_3_df)<-c('id','mean')

for (i in 1:length(id_loc_1_vals)){
    id_loc_1_df[i,1]<-id_loc_1_vals[i]
    id_loc_1_df[i,2]<-mean(filter(datam,geo_level_1_id == id_loc_1_vals[i])[,target_variable])
}
for (i in 1:length(id_loc_2_vals)){
    id_loc_2_df[i,1]<-id_loc_2_vals[i]
    id_loc_2_df[i,2]<-mean(filter(datam,geo_level_2_id == id_loc_2_vals[i])[,target_variable])
}
for (i in 1:length(id_loc_3_vals)){
    id_loc_3_df[i,1]<-id_loc_3_vals[i]
    id_loc_3_df[i,2]<-mean(filter(datam,geo_level_3_id == id_loc_3_vals[i])[,target_variable])
}
datam<-add_column(datam,geo_level_1_mean_damage = NA,.after='geo_level_1_id')
datam<-add_column(datam,geo_level_2_mean_damage = NA,.after='geo_level_2_id')
datam<-add_column(datam,geo_level_3_mean_damage = NA,.after='geo_level_3_id')
id_loc_1 <- match('geo_level_1_id', colnames(datam))
id_loc_2 <- match('geo_level_2_id', colnames(datam))
id_loc_3 <- match('geo_level_3_id', colnames(datam))
for (i in 1:nrow(datam)){
    datam[i,id_loc_1+1]<-filter(id_loc_1_df,id == datam[i,id_loc_1])[1,2]
    datam[i,id_loc_2+1]<-filter(id_loc_2_df,id == datam[i,id_loc_2])[1,2]
    datam[i,id_loc_3+1]<-filter(id_loc_3_df,id == datam[i,id_loc_3])[1,2]
}
#DO NOT FORGET processing for test values which zones could not be in training set

In [54]:
summary(datam)

  building_id      geo_level_1_id geo_level_1_mean_damage geo_level_2_id  
 Min.   :      4   Min.   : 0.0   Min.   :1.731           Min.   :   0.0  
 1st Qu.: 261190   1st Qu.: 7.0   1st Qu.:2.026           1st Qu.: 350.0  
 Median : 525757   Median :12.0   Median :2.172           Median : 702.0  
 Mean   : 525676   Mean   :13.9   Mean   :2.238           Mean   : 701.1  
 3rd Qu.: 789762   3rd Qu.:21.0   3rd Qu.:2.446           3rd Qu.:1050.0  
 Max.   :1052934   Max.   :30.0   Max.   :2.794           Max.   :1427.0  
                                                                          
 geo_level_2_mean_damage geo_level_3_id  geo_level_3_mean_damage
 Min.   :1.000           Min.   :    0   Min.   :1.000          
 1st Qu.:2.018           1st Qu.: 3073   1st Qu.:2.000          
 Median :2.210           Median : 6270   Median :2.200          
 Mean   :2.238           Mean   : 6258   Mean   :2.238          
 3rd Qu.:2.480           3rd Qu.: 9412   3rd Qu.:2.500          
 Max.   :3

In [5]:
dataNN <- recipe(damage_grade ~ ., datam) %>%
  step_nzv(everything(), -damage_grade) %>%
  step_range(count_floors_pre_eq, age, area_percentage, height_percentage min=0, max=1) %>%
  step_num2factor(damage_grade,levels=c('1','2','3')) %>%
  step_dummy(all_nominal(), one_hot = TRUE) %>%
  prep() %>%
  bake(new_data = NULL)